# ABench v0.3 — Runner and Evaluator Ecology

One-click execution of the **ABench v0.3 public development set** against OpenRouter,
with a blinded panel of evaluator models and code-owned exact measurements.

Built in the same shape as the GDA Phase 4C/4E notebooks in this repository:
append-only JSONL, resume by run key, typed missingness, and CSV summaries that
are the record.

**What this notebook does**

1. Fetches the harness (`abench_items.yaml`, `abench_metrics.py`, `abench_execute.py`).
2. Runs the built-in self-tests before spending any money.
3. Generates *n* independent runs per item per substrate model.
4. Scores every generation with an evaluator **ecology** — several models, not one judge.
5. Overwrites evaluator arithmetic with deterministic measurements from code.
6. Emits the seven reports Section 8 of the prompt pack requires.

**Claim ceiling.** ABench measures constrained comic construction: long-range
control, development, callback use, licensed transformation, isomorphic transfer
and termination discipline. It does not measure humour quality, audience
response, or cultural value. Section 8: never collapse the study into
"model X is funniest".

**Contamination.** This is a public development set, not a secure held-out
benchmark. `C-01` (unassisted recall) degrades first once these prompts
circulate. Record the access date; treat cross-date comparisons with suspicion.

## 1. Setup

In [ ]:
!pip install -q requests pyyaml pandas

import os, sys, subprocess, json, urllib.request
from pathlib import Path

REPO_RAW = "https://raw.githubusercontent.com/devinendorphin/alignment-friction-gda/main/abench"
HARNESS = ["abench_items.yaml", "abench_metrics.py", "abench_execute.py"]

# Use local copies when running inside a clone; otherwise fetch the released harness.
for name in HARNESS:
    if Path(name).exists():
        print(f"local  {name}")
        continue
    urllib.request.urlretrieve(f"{REPO_RAW}/{name}", name)
    print(f"fetched {name}")

sys.path.insert(0, str(Path.cwd()))

### 1.1 Self-tests before spending anything

Both modules carry assertions. The executor's set includes the blinding
guarantee for the transfer items (`X-01`, `X-02`): their evaluation context must
never name the source form.

In [ ]:
for module in ["abench_metrics.py", "abench_execute.py"]:
    result = subprocess.run([sys.executable, module, "--selftest"], capture_output=True, text=True)
    print(result.stdout.strip() or result.stderr.strip())
    assert result.returncode == 0, f"{module} self-test failed"

### 1.2 API key

Set `OPENROUTER_API_KEY`. In Colab, prefer the secrets panel (key icon) over
pasting into a cell. Skip this entirely if you only want `DRY_RUN = True`.

In [ ]:
try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    print("key loaded from Colab secrets")
except Exception:
    if not os.environ.get("OPENROUTER_API_KEY"):
        import getpass
        os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OPENROUTER_API_KEY (blank for dry run): ")
    print("key set" if os.environ.get("OPENROUTER_API_KEY") else "no key — dry run only")

## 2. Configure the run

`ITEM_SET` follows Section 5 of the prompt pack:

| set | items | runs | what it separates |
|---|---|---|---|
| `smoke` | `C-03` | 5 | one comparable output per model |
| `minimum` | `C-03`, `T-01`, `X-01` | 3 | canonical obedience vs transformation vs transfer |
| `full` | `R-01`, `C-01`, `C-02`, `C-03`, `T-01`, `T-02`, `X-01`, `X-02`, `S-01` | 5 | the stronger public-development run |

`C-04` (endurance, 1,800–2,400 words) is excluded from the named sets because it
is expensive; add it explicitly via `ITEM_IDS` when you want it.

Five runs per item is the comparable standard. Three is permitted and the
manifest then labels the study **exploratory** — that label travels with the data.

In [ ]:
ITEM_SET   = "minimum"      # "smoke" | "minimum" | "full"
ITEM_IDS   = ""             # e.g. "C-03,C-04" — overrides ITEM_SET when non-empty
RUNS       = None           # None -> 3 for "minimum", else 5
OUTDIR     = "abench_outputs"
DRY_RUN    = not bool(os.environ.get("OPENROUTER_API_KEY"))

# Manifest keys. Empty string = use every model in abench_items.yaml.
SUBSTRATE_MODELS = ""       # e.g. "claude_opus_4_7,gpt_5_2"
EVALUATOR_MODELS = ""       # e.g. "deepseek_r1,mistral_large,qwen_3_235b"

import yaml
manifest = yaml.safe_load(Path("abench_items.yaml").read_text())
print("substrate models:", list(manifest["models"]["substrate"]))
print("evaluator models:", list(manifest["models"]["evaluator"]))
print("item sets:", {k: v for k, v in manifest["item_sets"].items()})
print("\nDRY_RUN =", DRY_RUN)

**Edit the model tables in `abench_items.yaml` to match your account.** OpenRouter
model IDs move; a stale ID produces a `provider_error` row rather than a silent
gap, but a whole model of those is wasted spend.

One deliberate overlap: `llama_3_3_70b` appears in both tables by default. A model
scoring its own output is a known bias source — the reliability CSV reports each
evaluator's leniency against the panel median so you can see it. Drop it from
`EVALUATOR_MODELS` if you want a clean separation.

## 3. Execute

In [ ]:
cmd = [sys.executable, "abench_execute.py", "--outdir", OUTDIR]
if ITEM_IDS.strip():
    cmd += ["--item-ids", ITEM_IDS.strip()]
else:
    cmd += ["--set", ITEM_SET]
if RUNS:
    cmd += ["--runs", str(RUNS)]
if SUBSTRATE_MODELS.strip():
    cmd += ["--substrate-models", SUBSTRATE_MODELS.strip()]
if EVALUATOR_MODELS.strip():
    cmd += ["--evaluator-models", EVALUATOR_MODELS.strip()]
if DRY_RUN:
    cmd += ["--dry-run"]

print(" ".join(cmd), "\n")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()
assert process.returncode == 0, "run failed"

The run is **resumable**. Re-executing the cell skips any `run_id` already present
in `abench_raw_records.jsonl` and any evaluator call already present in
`abench_evaluator_outputs.jsonl`, so an interrupted run costs only what it had
not yet completed. Nothing is ever overwritten — Section 3 forbids selecting the
best output, repairing truncations, or deleting post-punchline material.

## 4. Load the results

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

out = Path(OUTDIR)
det   = pd.read_csv(out / "abench_deterministic_metrics.csv")
cons  = pd.read_csv(out / "abench_consensus_scores.csv")
mi    = pd.read_csv(out / "abench_model_item_summary.csv")
track = pd.read_csv(out / "abench_track_summary.csv")
cov   = pd.read_csv(out / "abench_coverage.csv")
flags = pd.read_csv(out / "abench_flag_frequency.csv")
rel   = pd.read_csv(out / "abench_evaluator_reliability.csv")
run_manifest = json.loads((out / "abench_run_manifest.json").read_text())

print(f"{len(det)} generations | study label: {run_manifest['study_label']} | dry run: {run_manifest['dry_run']}")
print(f"items: {run_manifest['items']}")
cons.head()

## 5. The seven required reports

Section 8: *"Never collapse the study immediately into 'model X is funniest'.
Report at least: recognition; canonical structural validity; transformation
validity; isomorphic transfer; refusal/coverage; rerun reliability; raw outputs."*

Each of the following is one of those seven.

### 5.1 Recognition (reported separately — never a substitute for performance)

In [ ]:
r01 = mi[mi.item_id == "R-01"]
if r01.empty:
    print("R-01 not in this run.")
else:
    display(r01[["model_key", "n_runs", "coverage_rate", "word_count_in_range_rate",
                 "mean_relational_invariant_control", "mean_global_coherence",
                 "mean_audience_model_control", "mean_novelty_depth"]]
            .sort_values("mean_relational_invariant_control", ascending=False))
    print("Recognition measures whether the model can describe the form. It says "
          "nothing about whether it can execute it — compare against 5.2 rather "
          "than reading it as a headline.")

### 5.2 Canonical structural validity

In [ ]:
canon = mi[mi.track == "canonical"]
if canon.empty:
    print("No canonical items in this run.")
else:
    cols = ["model_key", "item_id", "n_runs", "terminal_valid_rate", "terminal_required",
            "word_count_in_range_rate", "premature_leak_rate", "post_payoff_tail_rate",
            "mean_ending_discipline", "mean_patterned_development", "mean_global_coherence"]
    display(canon[[c for c in cols if c in canon.columns]].sort_values(["item_id", "model_key"]))
    print("terminal_valid_rate = label appears exactly once, as the final line, "
          "with no substantive words after it. Computed in code, not by an evaluator.")
    print("On C-01 the terminal rule is deliberately withheld, so terminal_required "
          "is False there and the rate is descriptive rather than a pass mark.")

### 5.3 Transformation validity

In [ ]:
trans = mi[mi.track == "transformation"]
if trans.empty:
    print("No transformation items in this run.")
else:
    cols = ["model_key", "item_id", "n_runs", "terminal_valid_rate",
            "mean_transformation_legitimacy", "mean_relational_invariant_control",
            "mean_persona_viewpoint_causality", "mean_novelty_depth"]
    display(trans[[c for c in cols if c in trans.columns]].sort_values(["item_id", "model_key"]))
    print("transformation_legitimacy asks whether the licensed deviation established "
          "and SUSTAINED a replacement constraint — not whether the output differed.")
    sdn = flags[(flags.flag == "SDN") & (flags.item_id.isin(trans.item_id.unique()))]
    if not sdn.empty:
        print("\nSDN (surface difference only) — the characteristic failure here:")
        display(sdn[["model_key", "item_id", "n_runs_flagged", "n_runs", "flag_rate"]])

### 5.4 Isomorphic transfer (blinded)

In [ ]:
xfer = mi[mi.track == "transfer"]
if xfer.empty:
    print("No transfer items in this run.")
else:
    cols = ["model_key", "item_id", "n_runs", "terminal_valid_rate",
            "mean_patterned_development", "mean_global_coherence",
            "mean_persona_viewpoint_causality", "source_form_named_rate"]
    display(xfer[[c for c in cols if c in xfer.columns]].sort_values(["item_id", "model_key"]))
    print("The evaluator context for X-01 and X-02 never names the source form; the "
          "executor raises rather than build such a prompt.")
    print("source_form_named_rate is the substrate naming it unprompted in its own "
          "output — a recognition signal, and a caution when reading these scores.")

### 5.5 Refusal and coverage (reported separately from structural failure)

In [ ]:
display(cov)
print("Section 3.10: refusals and content-filter interceptions are distinct "
      "outcomes, not low scores. A model that refuses is not a model that failed "
      "to construct — read coverage before reading any rating.")
nonzero = cov[(cov.refusal + cov.filter_interception + cov.provider_error) > 0]
if nonzero.empty:
    print("\nNo refusals, interceptions or provider errors in this run.")

### 5.6 Rerun reliability

In [ ]:
cols = ["model_key", "item_id", "n_runs", "rerun_sd_applicable_rating",
        "rerun_sd_word_count", "terminal_valid_rate"]
rr = mi[[c for c in cols if c in mi.columns]].copy()
display(rr.sort_values("rerun_sd_applicable_rating", ascending=False).head(20))
print("Large rerun SD means the item is not measuring a stable capacity in that "
      "model — one lucky generation is not a result. Compare it against any "
      "between-model gap before believing the gap.")
if "rerun_sd_applicable_rating" in mi.columns and mi.rerun_sd_applicable_rating.notna().any():
    print(f"\nMedian rerun SD: {mi.rerun_sd_applicable_rating.median():.3f}")

### 5.7 Raw outputs

In [ ]:
records = [json.loads(l) for l in (out / "abench_raw_records.jsonl").read_text().splitlines() if l.strip()]
by_id = {r["run_id"]: r for r in records}

INSPECT = cons.run_id.iloc[0]     # change to any run_id
rec = by_id[INSPECT]
print(f"{rec['run_id']}  |  {rec['model']['model_id']}  |  outcome: {rec['deterministic']['outcome']}")
print(f"words: {rec['deterministic']['observed_word_count']}  "
      f"label count: {rec['deterministic']['target_label_count']}  "
      f"terminal valid: {rec['deterministic']['canonical_terminal_valid']}")
print("=" * 78)
print(rec["raw_output"])

## 6. Auditing the evaluator ecology

The panel is an instrument, not ground truth. Three checks on it:

### 6.1 Panel reliability and leniency

In [ ]:
display(rel[rel.dimension == "ALL"][
    ["evaluator_key", "n_calls", "parse_success_rate", "schema_violations_per_parsed",
     "flags_per_parsed", "leniency_vs_panel", "mean_abs_deviation", "exact_agreement_rate"]])
print("leniency_vs_panel > 0 means the evaluator scores above the panel median. "
      "Persistent leniency is a calibration property of that model, not evidence "
      "about the substrates it scored.")

### 6.2 Can the evaluators count?

In [ ]:
wc = rel[rel.dimension == "ALL"][["evaluator_key", "mean_word_count_abs_error"]]
display(wc)
print("Mean absolute error between the evaluator's claimed word count and the true "
      "count. This is why Section 7 assigns counting to code: every numeric field "
      "in the released data comes from abench_metrics.py, and these evaluator "
      "guesses are retained only under evaluator_reported for this audit.")

### 6.3 Failure-flag landscape

In [ ]:
if flags.empty:
    print("No failure flags raised.")
else:
    pivot = flags.pivot_table(index="flag", columns="model_key", values="flag_rate",
                              aggfunc="mean").round(3).fillna(0)
    display(pivot)
    print("Flag glossary:")
    import abench_metrics as M
    for code_, name in M.FAILURE_FLAGS.items():
        if code_ in flags.flag.values:
            print(f"  {code_}  {name}")

## 7. Human review queue

Human rating remains primary; the ecology provides provisional scoring and flags
cases for review (Section 7). A generation enters the queue when any evaluator
asks for review, when the panel disagrees by two or more points on any dimension,
when evaluators disagree about the outcome, when an evaluator's outcome conflicts
with the code-computed one, or when nothing scored it at all.

In [ ]:
queue = cons[cons.requires_human_review].copy()
queue = queue.sort_values(["max_panel_range"], ascending=False)
qcols = ["run_id", "item_id", "model_key", "outcome", "canonical_terminal_valid",
         "observed_word_count", "word_count_in_range", "mean_applicable_rating",
         "max_panel_range", "evaluator_outcome_disagreement",
         "evaluator_outcome_conflicts_with_code", "majority_failure_flags"]
queue[[c for c in qcols if c in queue.columns]].to_csv(out / "abench_human_review_queue.csv", index=False)
print(f"{len(queue)} of {len(cons)} generations queued for human review "
      f"({len(queue)/max(len(cons),1):.1%}) -> abench_human_review_queue.csv")
display(queue[[c for c in qcols if c in queue.columns]].head(15))

## 8. Reading the run

The diagnostic question is **where each model's competence lives and where it
breaks**, not which model wins. Some shapes worth looking for:

- **Recognition without execution** — high `R-01` ratings, low `C-03`
  `terminal_valid_rate`. The model can describe the form it cannot hold.
- **Canonical obedience without transfer** — clean `C-03`, collapsing `X-01`.
  The structure was retrieved, not constructed.
- **Transformation as relabelling** — `T-01` terminal line lands, but
  `transformation_legitimacy` is low and `SDN` fires. The replacement constraint
  was announced rather than sustained.
- **Endurance decay** — `C-04` (if run) shows `LD` list drift and repeated events
  under changed nouns where `C-03` was clean. Long-range control has a ceiling.
- **Termination failure as a separate axis** — `PPC` and `PTL` are frequent and
  largely independent of construction quality. Ending discipline is its own skill.
- **Refusal is not failure** — read `abench_coverage.csv` first. A high refusal
  rate makes every rating for that model a conditional statement about the
  subset that got through.

`mean_applicable_rating` exists for within-item comparison and review triage. It
is not a leaderboard, and a run reported as a single ranked column is a run
reported wrongly.